> Download and tidy up *real* data for training.

In [ ]:
import pathlib
import os

import torch

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.data
import trapallada.plot

So that the notebook is aware of modifications in the the `import`ed modules

In [ ]:
%load_ext autoreload
%autoreload 2

The directory where observations' data will be saved

In [ ]:
download_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
download_directory.mkdir(exist_ok=True)

That in which data will be saved

In [ ]:
output_directory = pathlib.Path.cwd().parent / 'data'
assert output_directory.exists()

## Data for Javi's paper

In [ ]:
javi_obs = '0605950101'
javi_obs_dir = trapallada.xsa.util.download_extract_filter(obs=javi_obs, output_directory=download_directory)
javi_obs_dir

The downloaded files

In [ ]:
fits_files = list(javi_obs_dir.glob('*.FTZ'))
fits_files[:3]

In [ ]:
df, skipped_files_list = trapallada.xsa.util.read_fits_files(fits_files)
print(f'len{skipped_files_list=}')
df.head()

We have `number of rows` measurements for each one of the `number of columns` signals

In [ ]:
df.shape

The period is

In [ ]:
df.index[1] - df.index[0]

Signals are *binned* so that a bunch of samples/measurements are added up together

In [ ]:
bin_size = 63
# bin_size = 5
binned_df = trapallada.xsa.util.bin_fits_df(df, bin_size=bin_size)
binned_df

The period is now

In [ ]:
binned_df.index[1] - binned_df.index[0]

Signals might have `NaN`s

In [ ]:
signals = binned_df.values.T
np.isnan(signals).sum()
# np.argwhere(np.isnan(signals))

if there are `NaN`s, they are fixed by interpolation

In [ ]:
if binned_df.isna().any().any():
    binned_df = binned_df.interpolate(method='linear', axis=0)

In [ ]:
n_samples_to_plot = 2_000
sample_signal = binned_df['P0605950101PNS003SRCTSR8001.FTZ'].iloc[:n_samples_to_plot].values
trapallada.plot.signal(sample_signal)

Output file...

In [ ]:
output_file = (output_directory / f'{javi_obs}_bin_size_{bin_size}').with_suffix('.pt')
output_file

...to save the signals (notice we add an extra *dummy* axis)

In [ ]:
torch.save(np.expand_dims(binned_df.values.T, 1), output_file)

A `SignalsDataset` is instantiated...

In [ ]:
ds = trapallada.data.SignalsDataset(output_file, metadata_filename=None, normalize=True)
len(ds)

...and plotted

In [ ]:
trapallada.plot.signals_dataset(ds, columns=[pathlib.Path(c).stem for c in df.columns], n_signals=None, n_samples=1_000)

A particular signal

# Dataset

**All** the observations in XSA whose duration is in the given interval are downloaded

In [ ]:
# trapallada.xsa.util.download_observations(download_directory=download_directory, duration_interval=(35_000, 45_000))

The *tar* files in the download directory

In [ ]:
tar_files = list(download_directory.glob('*.tar'))
tar_files[:3]

In [ ]:
fits_df = trapallada.xsa.util.read_tar(tar_files[2], max_n_sources=4)
fits_df

In [ ]:
fits_df.index[1] - fits_df.index[0]

If there is *any* `NaN` at all (in any column, at any time instant), it's filled in using interpolation.

In [ ]:
if fits_df.isna().any().any():
    print('Interpolating...')
    fits_df = fits_df.interpolate(method='linear', axis=0)

In [ ]:
bin_size = 63
# bin_size = 32
binned_df = trapallada.xsa.util.bin_fits_df(fits_df, bin_size=bin_size)
binned_df

In [ ]:
period = binned_df.index[1] - binned_df.index[0]
period

In [ ]:
trapallada.plot.df(binned_df, columns=[pathlib.Path(e).stem for e in binned_df.columns], index_type='T')

In [ ]:
sliced_signals, sliced_times = trapallada.xsa.util.slice(binned_df, window_size=90, step=50, tail_as_window=False)
sliced_signals.shape

In [ ]:
sliced_signals[3]

In [ ]:
trapallada.plot.signals_array(sliced_signals)

In [ ]:
signals, metadata_df = trapallada.xsa.util.tars_to_array(tar_files[:5])
# signals, metadata_df = trapallada.xsa.util.tars_to_array(tar_files)

In [ ]:
metadata_df.head()

The base name for the output files...

In [ ]:
output_file_stem = 'sample_35000_45000'

...to be saved

In [ ]:
torch.save(torch.from_numpy(np.expand_dims(signals, 1)), (output_directory / (output_file_stem + '_signals')).with_suffix('.pt'))
metadata_df.to_parquet((output_directory / (output_file_stem + '_metadata')).with_suffix('.parquet'))

A *PyTorch* `Dataset` is built

In [ ]:
ds = trapallada.data.SignalsDataset(
    (output_directory / (output_file_stem + '_signals')).with_suffix('.pt'),(output_directory / (output_file_stem + '_metadata')).with_suffix('.parquet'), normalize=True)

A particular sample

In [ ]:
i = 10
print(ds[i])
trapallada.plot.signal(ds[i][1][0])